In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/test/RPA-6542")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] [Untitled]_23-48


In [3]:
IDX_FILE = 0

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\PP-OCRv5_server_det')
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\cyrillic_PP-OCRv5_mobile_rec')
The specified device (GPU) is not available! Switching to CPU instead.
2026-07-20 12:41:26.938 | INFO     | vision_core.pipelines.build_document:build:105 - Обработка страницы 0 с dpi 200...
2026-07-20 12:41:27.045 | INFO     | vision_core.pipelines.build_document:_process_page:187 - Коррекция ориентации и наклон

In [6]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-07-20 12:41:47.665 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 926 символов из 1 страниц
2026-07-20 12:41:47.666 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:93 - summary_cell_texts: ['По данным Акционерное общество "Уральская фольга", ИНН 6646010043', 'ПО ДАННЫМ ОБЩЕСТВО С ОГРАНИЧЕННОИ ОТВЕТСТВЕННОСТЬЮ "КРАСНОЯРСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД", ИНН 246504З748']
2026-07-20 12:41:47.667 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: 01.01.2025 - 31.10.2025 между Акционерное общество "Уральская фольга", ИНН 6646о1004з И ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "КРАСНОЯРСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД", ИНН 2465О4З748 Мы, нижеподписавшиеся, Акционерное общество "Уральская фольга", ИНН 6646010043 , с одной стороны, ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "КРАСНОЯРСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД". ИНН 2465О4З748 , с другой стороны, соста

In [7]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НАЧАЛЬНОЕ', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='2', id_col=2, buyer_col=6)), LedgerEntry(record='САЛЬДО НАЧАЛЬНОЕ ПО ДОГОВОРУ', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='3', id_col=2, buyer_col=6)), LedgerEntry(record='ПРИОБРЕТЕНИЕ ТОВАРОВ И УСЛУГ (ЦО0972 ОТ 28.02.2025) (4 721 093,96 RUB)', value=0.0, date='10.03.2025', row_reference=RowReference(id_table='0', id_row='4', id_col=2, buyer_col=6)), LedgerEntry(record='ПЛАТЕЖНОЕ ПОРУЧЕНИЕ ИСХОДЯЩЕЕ (01149 OT 20.03.2025) (4 721 093,96 RUB)', value=4721093.96, date='20.03.2025', row_reference=RowReference(id_table='0', id_row='5', id_col=2, buyer_col=6)), LedgerEntry(record='ПРИОБРЕТЕНИЕ ТОВАРОВ И УСЛУГ (ЦО2731 ОТ 22.05.2025 ) (4 292 037,69 RUB)', value=0.0, date='27.05.2025', row_reference=RowReference(id_table='0', id_row='6', id_col=2, buyer_col=6)), LedgerEntry(record='ПЛАТЕЖНОЕ ПОРУЧЕНИЕ ИСХОДЯЩЕЕ (02430 OT 11.06.2025) (4 292 037,69 RUB)', value

In [8]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = f"""
            По данным покупателя {data.buyer}
            По данным продавца {data.seller}
            В период: {data.period.start} - {data.period.end}
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-07-20 12:41:47.882 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: D:\projects\rusal_recon_srv\repo\recon_vision\assets\fonts\LiberationSerif-Regular.ttf
2026-07-20 12:41:48.049 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=1653x2339 aligned=1653x2339 canvas=1653x2339
2026-07-20 12:41:48.053 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R2:C6 значение=0.0
2026-07-20 12:41:48.054 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_font:229 - загружаем шрифт из D:\projects\rusal_recon_srv\repo\recon_vision\assets\fonts\LiberationSerif-Regular.ttf для размера 18
2026-07-20 12:41:48.055 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R3:C6 значение=0.0
2026-07-20 12:41:48.056 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R4:C6 значение=0.0
2026-07-20 12:41:48.057

In [9]:
out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
Path(out_path).write_bytes(filled_pdf)
print(out_path)

../examples/output/18470938_filled.pdf
